# Structured Prediction in the Wild with Chameleon

<div align=center>

[KAIST AIC.20000: Introduction to AI Computing (Fall 2026)](https://intro-aic.kaist.ac.kr/)

Starter Notebook — Few-Shot Dense Visual Prediction with Chameleon

Instructor: Seunghoon Hong

TA: Semin Kim (seminkim [at] kaist.ac.kr)

(Source: [Chameleon, ECCV 2024 oral](https://arxiv.org/abs/2404.18459) · [official code](https://github.com/GitGyun/chameleon))

<img src="https://raw.githubusercontent.com/GitGyun/chameleon/main/main_figure.png" width="90%"></img>
</div>

Welcome! This notebook walks through **structured prediction**: computer-vision problems whose
answer is not a label but a *map* — a value for every pixel.

You will fine-tune a single pretrained model, **Chameleon**, on a new dense-prediction task using
about 20 labelled images, then watch the same model solve five very different tasks. By the end
you should be able to take a problem of your own, decide how to encode it as a dense output, and
train a predictor for it.

## 1. What is structured prediction?

Most introductory vision tasks map an image to **one** answer: *"is this a cat?"*. Structured
prediction maps an image to **an answer with spatial structure** — one that has as many parts as
the image has pixels, and whose parts are not independent.

| task | output |
| --- | --- |
| semantic segmentation | a class for every pixel |
| keypoint detection | one heatmap per body joint |
| depth estimation | a distance for every pixel |
| object counting | a density map whose *sum* is the count |
| instance segmentation | vector fields that point toward each object's centre |
| surface normals | a 3-vector per pixel |

The list looks heterogeneous, but every entry is the same shape of problem: **predict a
`(C, H, W)` tensor from one or more `(3, H, W)` images**, where the meaning of the `C` channels is
whatever you decide. That single observation is what makes a *generalist* possible.

Why this matters in practice: real problems rarely come with a million labelled examples. A
factory wants to find a new kind of defect; a biologist wants to segment a cell type nobody has
annotated. You may have twenty labelled images and one week. The question is whether that is
enough.

## 2. How Chameleon works

Chameleon is a **few-shot generalist**: instead of learning one task, it learns *how to infer a
task from examples*.

At inference it is given two things:

- a **support set** — a handful of `(image, label)` pairs that define the task, and
- a **query image** — the one you want a prediction for.

```
support images  ──> image encoder ──┐
                                    ├──> matching ──> label decoder ──> prediction
support labels  ──> label encoder ──┤                      ▲
query image     ──> image encoder ──┘──────────────────────┘
```

1. The **image encoder** (a BEiT-v2-Large ViT) embeds the support images and the query image.
2. The **label encoder** (a ViT-Large) embeds the support *labels*, treating each label channel as
   a one-channel image.
3. The **matching module** compares the query's patch tokens against the support images' patch
   tokens, and uses the resulting attention to mix the support *label* tokens. Patches that look
   alike should get alike labels.
4. The **label decoder** (a DPT head) turns the mixed label tokens back into a dense map.

The model was **meta-trained** on a large mixture of dense tasks (Taskonomy, COCO, MidAir, MPII,
DeepFashion, FreiHand), never on the five tasks below. Adapting it to a new task is
**stage 1 fine-tuning**, and it updates only a tiny slice of the network:

- every **bias** term in the image encoder (the [BitFit](https://arxiv.org/abs/2106.10199) trick),
- the matching module's temperature `alpha`, and
- one LayerNorm.

That is well under 1% of the parameters, which is why each task checkpoint is ~20 MB rather than
~3 GB, and why 20 images are enough to avoid catastrophic overfitting.

**The three stages.** You will see `stage` in every config:

| stage | meaning | do you run it? |
| --- | --- | --- |
| 0 | meta-training on the task mixture | no — we give you the result (`generalist.ckpt`) |
| 1 | fine-tuning on one downstream task | **yes**, in Step 4 |
| 2 | inference/evaluation with a fine-tuned checkpoint | **yes**, in Steps 5–7 |

## **What You Will Do in This Notebook** 🚨

1. **Set up** the environment and mount the shared checkpoints and demo data. *(Step 1)*
2. **Look at the input and label format** — every task comes down to the same three tensors —
   using animal keypoints as the worked example. *(Step 2)*
3. **Load the meta-trained model** and see exactly which parameters fine-tuning touches. *(Step 3)*
4. **Fine-tune it yourself** on 20 giraffe images, with the training loop written out. *(Step 4)*
5. **Run inference and visualise** your result, and compare against the reference checkpoint.
   *(Step 5)*
6. **Run all five tasks** and see the same model produce five kinds of dense output. *(Step 6)*
7. **Start your project**, optionally on a bonus track (+2 pt): fine-tune on your own task with
   the folder-based template, or build an application around the model. *(Step 7)*

**NOTE**: search for `TODO` (Ctrl+F) — those are the places you need to fill in or change.

---
## Step 1: Set Up Your Environment

If you opened this notebook with the **Open in Colab** badge on the GitHub page, first do
**File → Save a copy in Drive**, so your changes are kept. Then go to `Runtime` →
`Change runtime type` and choose **T4 GPU** under Hardware Accelerator.

The cell below should print `True` followed by the GPU status.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi

Next, install the libraries Chameleon needs. Colab already ships `torch`, `torchvision`, `numpy`,
`scipy`, `matplotlib` and `opencv` built against its own CUDA, so we deliberately do **not**
reinstall them.

Run this once per Colab runtime. If Colab asks you to restart the runtime afterwards, restart it
and continue from the next cell.

In [ ]:
# Pin NumPy to the version this runtime has already loaded: if a dependency upgraded it mid-session,
# the running kernel would mix two NumPy versions and fail until restarted.
import numpy
NUMPY_PIN = f"numpy=={numpy.__version__}"

%pip install -qq pytorch_lightning==2.4.0 timm==0.6.13 easydict einops \
                 scikit-image pycocotools fastremap numba flow_vis "$NUMPY_PIN"

Now get the starter code: the official Chameleon implementation plus a small `starter_utils`
module with the notebook's plumbing. On Colab, the cell below clones this notebook's GitHub
repository; if you run the notebook locally from a clone of the repository, it uses that clone.

> Every change we made to the official code is listed in `chameleon/PATCHES.md`.

In [ ]:
import os, sys

REPO_URL = "https://github.com/GitGyun/AIC200-Project-StructuredPrediction.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = "/content/AIC200-Project-StructuredPrediction"
    if not os.path.isdir(REPO_DIR):
        !git clone -q --depth 1 $REPO_URL $REPO_DIR
else:
    REPO_DIR = "."    # running locally from the repository root

CHAMELEON_DIR = os.path.join(REPO_DIR, "chameleon")
if CHAMELEON_DIR not in sys.path:
    sys.path.insert(0, CHAMELEON_DIR)

import starter_utils as su
print("starter code ready:", CHAMELEON_DIR)

### Mount Google Drive

The meta-trained checkpoint is ~2.8 GB, so we read it from a **shared Google Drive folder** rather
than downloading it into every session.

1. Open the [shared course folder](https://drive.google.com/drive/folders/1sw2E-ocZmLo5ph9ZdvgHzk7uIhYDndb6?usp=drive_link) (also linked from `README.md`).
2. Click **Add shortcut to Drive** → **My Drive**, keeping the folder's name
   (`AIC200-StructuredPrediction`).
3. Run the cell below and grant access when prompted.

Reading straight from Drive means nothing is re-downloaded when your runtime resets. Your own
outputs go to a separate folder in your **own** Drive, since the shared folder is read-only.

*Running locally instead?* Put the `checkpoints/` and `demo_data/` folders (or symlinks to them)
in a `drive/` folder next to this notebook.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/AIC200-StructuredPrediction"   # the shared folder's shortcut
    WORK_DIR   = "/content/drive/MyDrive/AIC200-my_work"                # yours, writable
else:
    DRIVE_ROOT = "drive"
    WORK_DIR   = os.path.join(DRIVE_ROOT, "my_work")

CHECKPOINT_ROOT = os.path.join(DRIVE_ROOT, "checkpoints")
DEMO_DATA_ROOT  = os.path.join(DRIVE_ROOT, "demo_data")

# Your own outputs (fine-tuned checkpoints, figures) go here and survive runtime resets.
os.makedirs(WORK_DIR, exist_ok=True)

print("checkpoints :", CHECKPOINT_ROOT)
print("demo data   :", DEMO_DATA_ROOT)
print("your outputs:", WORK_DIR)

### Check the GPU and the shared assets

The cell below prints your GPU, writes `data_paths.yaml` (the one file Chameleon reads to find each
dataset), and checks that every checkpoint and dataset the notebook needs is present.

Inference (Steps 5–7) reports the peak GPU memory of **support encoding** and of **inference**,
e.g. `[GPU] inference: peak 11.48 GB / 14.7 GB (fits a T4)`. 14.7 GB is what PyTorch can use on a
Colab T4. If you change a setting and a line reports `EXCEEDS a T4`, it will run out of memory on
Colab even if your own GPU is larger.

Everything in this notebook runs in fp32. The released configs say `bf16`, which a T4 does not
support, so `starter_utils` overrides that for you.

In [ ]:
su.describe_gpu()
print()

su.write_data_paths(DEMO_DATA_ROOT, chameleon_root=CHAMELEON_DIR)
print()

su.check_assets(CHECKPOINT_ROOT, DEMO_DATA_ROOT)

In [ ]:
META_CKPT = su.find_meta_trained(CHECKPOINT_ROOT)
print("meta-trained checkpoint:", META_CKPT)
print()

for task, spec in su.TASKS.items():
    names = su.list_checkpoints(CHECKPOINT_ROOT, task)
    print(f"{spec['title']}")
    print(f"   dataset   : {spec['dataset']}")
    print(f"   output    : {spec['output']}")
    print(f"   available : {', '.join(names)}")
    print(f"   default   : {spec['default_ckpt']}")

---
## Step 2: Input and Label Format

Everything Chameleon does is expressed with **three tensors per example**:

| tensor | shape | meaning |
| --- | --- | --- |
| `X` | `(3M, H, W)` | the input: `M` images of the same scene stacked along channels, float in `[0, 1]` |
| `Y` | `(C, H, W)` | the dense label, float in `[0, 1]` — one channel per output map |
| `M` | `(C, H, W)` | 1 where `Y` is a valid target, 0 where the label is unknown |

(Here `M` in `3M` counts input images. It is a different `M` from the mask tensor.)

**Multi-modal inputs.** Some tasks need more than one image per example. Chameleon splits `X` into
its `M` three-channel images, embeds each with the same RGB-pretrained backbone, and lets the
tokens attend to each other. So the channel count must be a multiple of 3: a single-channel
modality (a grayscale stain, a depth map) is repeated to 3 channels. The released weights take
up to `M = 2`.

`M` is the part people forget. It is how you say *"this joint is occluded"* or *"the depth sensor
returned nothing here"*, so the loss ignores those pixels instead of training on garbage.

Here is how the five tasks fill in `X` and `Y`:

| task | input images (`3M`) | `C` | what each label channel holds |
| --- | --- | --- | --- |
| animal keypoints | RGB photo (3) | 17 | a Gaussian blob at one joint; mask 0 for joints not visible |
| video object segmentation | RGB frame (3) | #instances | a binary mask for one tracked instance |
| medical lesion segmentation | RGB dermoscopy (3) | 1 | a binary lesion mask |
| object counting | RGB photo + exemplar map (6) | 1 | a density map whose sum is the object count |
| cell instance segmentation | cytoplasm + nuclei stains, each gray→3 (6) | 3 | two flow channels toward the cell centre + a foreground mask |

**Your own task will be a new row in that table.** Deciding what goes in `Y` *is* the modelling
work.

Let's look at real data. We will use the **AP-10K** animal-keypoint task and the species
**Giraffe** throughout Steps 2–5.

The demo data uses one layout for every task — numbered files, no train/test split:

```
demo_data/ap10k/
├── images/   0.jpg 1.jpg ... 29.jpg
└── labels/   0.npy 1.npy ... 29.npy     # (17, 3): x, y, visibility of each joint
```

The first `shot` files are the **support set** (here 20), and the rest are **queries**. Each
dataset class turns one `(image, label file)` pair into `X, Y, M`.

In [ ]:
TUTORIAL_TASK    = "animal_keypoint_detection"
TUTORIAL_SPECIES = "Giraffe"   # the demo data holds Giraffe; other species need the full AP-10K

# A config is all the settings Chameleon needs. Built in memory -- no experiment directories,
# no logging, nothing outside this session.
data_cfg = su.build_config(
    stage=1, dataset="ap10k",
    class_name=TUTORIAL_SPECIES,
    shot=20,           # size of the support set
    support_idx=0,     # which slice of the labelled pool to use as support
)
# the loader for demo_data/ap10k
AP10KDataset = su.dataset_class(data_cfg)

# eval_mode=True  -> no random augmentation, and the dataset also returns bookkeeping info
# resize=True     -> squash the whole image to img_size (what support examples want)
support_set = AP10KDataset(
    data_cfg, split="train",
    base_size=data_cfg.base_size, crop_size=data_cfg.img_size,
    eval_mode=True, resize=True, dset_size=data_cfg.shot,
)

print(f"support set: {len(support_set)} labelled {TUTORIAL_SPECIES} images")
X, Y, M, aux = support_set[0]
print(f"  X {tuple(X.shape)}  image,  range [{X.min():.2f}, {X.max():.2f}]")
print(f"  Y {tuple(Y.shape)}  label,  range [{Y.min():.2f}, {Y.max():.2f}]")
print(f"  M {tuple(M.shape)}  mask,   {int(M[:, 0, 0].sum())} of {M.shape[0]} joints are visible")

Look at the support images themselves. These twenty images are the *entire* task definition — this
is everything the model gets to learn "giraffe keypoints" from.

In [ ]:
su.show_images(
    [support_set[i][0] for i in range(min(10, len(support_set)))],
    titles=[f"support {i}" for i in range(10)],
    title=f"AP-10K support set — {TUTORIAL_SPECIES} (showing 10 of {len(support_set)})",
    cols=5,
)

Now open up `Y` for a single image. The 17 channels are not an abstraction — each one is literally
a small blob drawn at one joint.

Channels that are entirely black are joints the annotator could not see; the corresponding `M`
channel is 0, so they contribute nothing to the loss.

In [ ]:
su.show_channels(
    Y, titles=AP10KDataset.CLASS_NAMES,
    title=f"Y for one {TUTORIAL_SPECIES} image — 17 channels, one Gaussian blob per joint",
    cols=6,
)

visible = [n for n, v in zip(AP10KDataset.CLASS_NAMES, M[:, 0, 0]) if v > 0]
print(f"visible joints ({len(visible)}): {', '.join(visible)}")

Overlaid on the image, those 17 blobs become a skeleton. This is the same `Y`, drawn for humans.

In [ ]:
import torch

batch = [support_set[i] for i in range(min(5, len(support_set)))]
X_b = torch.stack([b[0] for b in batch]).float()
Y_b = torch.stack([b[1] for b in batch]).float()
kp_b = torch.stack([b[3][0] for b in batch])

from downstream.ap10k.utils import vis_animal_keypoints
from dataset.utils import dense_to_sparse

overlays = []
for i in range(len(X_b)):
    img = (X_b[i] * 128).byte().permute(1, 2, 0).numpy().copy()
    kps = dense_to_sparse(Y_b[i]).transpose(0, 1).numpy()
    kps[2] = kp_b[i, :, 2].float().numpy()          # visibility from the annotation
    overlays.append(torch.from_numpy(vis_animal_keypoints(img, kps)).permute(2, 0, 1) / 255)

su.show_images(overlays, title="the same labels, drawn as a skeleton", cols=5)

> **Pause here if you are designing your own task.** Ask yourself: what are the channels of my
> `Y`? What does one channel *mean*? Where is my label genuinely unknown, so `M` should be 0?
> Getting this right matters far more than any hyperparameter later on.

---
## Step 3: Load the Meta-Trained Model

`load_model` does three things at once: it builds the architecture, loads the meta-trained
weights, and merges the configs (meta-trained → fine-tuned → yours, so **your** settings win).

It is also what promotes `n_input_images` to 2, because the generalist was meta-trained with
paired (stereo) inputs. That is why we go through `load_model` rather than constructing the model
by hand — building it yourself would silently produce a model whose weights do not line up.

Loading ~2.8 GB from Drive takes a minute or two.

In [ ]:
finetune_cfg = su.build_config(
    stage=1, dataset="ap10k",
    class_name=TUTORIAL_SPECIES,
    shot=20,
    # --- T4 memory budget. The paper uses global_batch_size=8, channel_sampling=12, n_steps=10000
    # --- on a 40GB GPU. These two settings peak at ~13 GB in fp32 -- they fit a T4, barely.
    global_batch_size=2,    # episodes per step (split internally into support + query)
    channel_sampling=6,     # how many of the 17 joints to train on per step
    n_steps=200,            # TODO: raise this if you have time/compute
    lr=0.001,
)

model, finetune_cfg = su.load_chameleon(finetune_cfg, META_CKPT)
print("\nmodel ready on", next(model.parameters()).device)

Which parameters does fine-tuning actually move?

In [ ]:
su.trainable_parameter_report(model)

---
## Step 4: Fine-Tune on the Support Data

Chameleon fine-tunes **episodically**. Each training step:

1. draws a small batch of labelled images,
2. splits them into a pseudo-support half and a pseudo-query half,
3. hides the query labels, predicts them from the support half, and
4. takes the loss between prediction and the hidden truth.

So the model is never trained to memorise giraffes — it is trained to *use examples of giraffes*.
That is why the same procedure works for a task it has never seen.

The loss is task-specific and lives on the learner. For keypoints it is a **spatial softmax
cross-entropy**: each channel is a distribution over pixel locations, and we maximise the
probability at the true joint.

Everything below is a plain PyTorch loop — no Trainer, no logger, no checkpoint callbacks.

In [ ]:
from tqdm.auto import trange
from train.optim import get_optimizer
from dataset.utils import to_device

optimizer, lr_scheduler = get_optimizer(finetune_cfg, model.model)
train_loader = model.learner.get_train_loader()

model.train()
losses, data_iter = [], iter(train_loader)

for step in trange(finetune_cfg.n_steps, desc="fine-tuning"):
    try:
        batch = next(data_iter)
    except StopIteration:                      # the loader is finite; just start it again
        data_iter = iter(train_loader)
        batch = next(data_iter)
    batch = to_device(batch, "cuda")

    # model(batch) splits the batch into support/query and runs the full matching pipeline
    Y_pred, Y_query, M_query, _ = model(batch)
    loss, _ = model.learner.compute_loss(Y_pred, Y_query, M_query)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    lr_scheduler.step(step)                    # constant schedule for this task, but be explicit

    losses.append(loss.item())

model.eval()
su.plot_loss(losses, title=f"fine-tuning on {finetune_cfg.shot} {TUTORIAL_SPECIES} images")
print(f"loss: {losses[0]:.4f} (start) -> {sum(losses[-10:]) / 10:.4f} (last 10 steps)")

Save the result. Only the fine-tuned slice is stored, which is why the file is ~20 MB rather than
~3 GB — and it is written in exactly the format the official checkpoints use, so everything that
reads theirs can read yours.

In [ ]:
MY_CKPT = os.path.join(WORK_DIR, f"my_{TUTORIAL_SPECIES.lower()}.ckpt")
su.save_finetuned(model, MY_CKPT)

# drop every reference into the training graph, or ~5 GB stays allocated and Step 5 runs out of memory
del model, optimizer, lr_scheduler, train_loader, data_iter, batch, Y_pred, Y_query, M_query, loss
su.free_gpu()

---
## Step 5: Inference and Visualisation

Inference is stage 2, and it has two phases:

1. **Encode the support set once.** `encode_support` pushes the support images and labels through
   the encoders and caches the tokens. This is the expensive part, and it happens once per task.
2. **Predict each query.** `inference` embeds the query, matches it against the cached support
   tokens, decodes, and post-processes.

Every task's figure has the same three parts, one column per query:

1. **the input channels** — one row per image in `X` (two rows for the two-image tasks);
2. **the raw dense label and prediction** — the `(C, H, W)` tensors the model is actually trained
   on and outputs: heatmaps, soft masks, density maps, flow fields;
3. **the task-level label and prediction** — what those maps *mean*: a skeleton, a mask, a count,
   cell instances. These are decoded with the same post-processing the evaluation code uses.

The gap between parts 2 and 3 is the point of this project: the model only ever sees dense maps,
and turning a real-world output into such a map (and back) is the modelling decision.

Notice that we never tell the code the image resolution or the shot count by hand:
`derive_stage2_overrides` reads them out of the checkpoint itself. Every checkpoint carries the
config it was produced with, which is what keeps inference consistent with fine-tuning.

In [ ]:
def run_task(task, ckpt_name=None, n_query=4, eval_shot=None, checkpoint_path=None, verbose=True):
    '''Load a fine-tuned checkpoint, predict on demo queries, and show the result.'''
    spec = su.TASKS[task]

    ft_ckpt = checkpoint_path or su.find_finetuned(CHECKPOINT_ROOT, task, ckpt_name)
    overrides = su.derive_stage2_overrides(ft_ckpt, verbose=verbose)
    if eval_shot is not None:
        # fewer support examples = less GPU memory. Set `shot` too: tasks that tile their images
        # (`autocrop`, e.g. Cellpose) take the support-set size from `shot` and ignore `eval_shot`.
        overrides["shot"] = overrides["eval_shot"] = eval_shot
    overrides["eval_size"] = n_query            # load only the queries we are going to show

    cfg = su.build_config(stage=2, dataset=spec["dataset"], **overrides)
    model, cfg = su.load_chameleon(cfg, META_CKPT, ft_ckpt, verbose=False)
    model.eval()

    # the support set is encoded once, then every query reuses the cached tokens; peak GPU memory
    # of both phases is printed. Each sample holds X, Y, M, aux and `raw`, the dense prediction.
    samples = su.predict_queries(model, n_query, support_verbose=verbose)

    su.show_predictions(model, samples, title=f"{spec['title']} — {os.path.basename(ft_ckpt)}")
    return model, cfg, samples

First, the **reference checkpoint** — Chameleon fine-tuned properly on 20 giraffe images.

In [ ]:
model, cfg, _ = run_task(TUTORIAL_TASK, ckpt_name=TUTORIAL_SPECIES, n_query=4)
del model
su.free_gpu()

Now **your own 200-step checkpoint** from Step 4, for comparison.

Expect it to be worse — 200 steps is roughly 2% of the reference run's budget. What is worth
noticing is *how* it is worse: which joints it already places well, and which it has not learned.

In [ ]:
model, cfg, _ = run_task(TUTORIAL_TASK, checkpoint_path=MY_CKPT, n_query=4)
del model
su.free_gpu()

---
## Step 6: Five Tasks, One Model

The same weights, the same `inference` call, five different kinds of dense output. The only thing
that changes is the support set and the ~20 MB of fine-tuned biases.

Each cell is independent, so you can run just the ones you care about. Restart the runtime if you
hit an out-of-memory error partway through.

In [ ]:
# Support-set sizes for the demos. Smaller than the paper's settings where a 16GB T4 cannot hold
# the full support set -- the official value for each task is printed by `derive_stage2_overrides`.
DEMO_EVAL_SHOT = {
    "animal_keypoint_detection":   None,   # 20-shot fits; all 17 joints need the coverage
    "video_object_tracking":       None,   # 1-shot by construction
    "medical_lesion_segmentation": 10,     # official 50 @ 384px does not fit a T4
    "object_counting":             8,      # official 50 @ 512px does not fit a T4
    "cell_instance_segmentation":  10,     # official 50, with sliding-window cropping
}

### 6.1 Animal Keypoint Detection (AP-10K)

- **Centring the animal (object-level cropping).** Keypoint detection here is *per animal*: the
  model never has to find the animal or tell two animals apart, because each input is already one
  centred animal.
  - *How it is obtained:* every AP-10K annotation comes with a **bounding box drawn by the
    annotators**. An offline script (`crop_and_resize.py` in the dataset folder) enlarges each box
    1.2× around its centre, clamps it to the image, crops it, and resizes it to 256×256 (aspect
    ratio is not preserved). That gives one image per annotated animal, named
    `{image_id}_{annotation_id}.jpg`. Photos containing more than one animal are skipped (the
    `skip_crowd` index in `dataset/meta_info/ap10k/`).
  - *How it is used:* the joint coordinates are mapped through the same crop and resize
    (`AP10KDataset._load_keypoint`), so labels and pixels stay aligned. On a new photo there is no
    annotated box, so the crop has to come from an **animal detector** first; the detector is
    part of the system, not of this model.
- **Preprocessing.** Support crops are resized to the model's 224×224; queries are predicted at
  256×256. Fine-tuning takes random 224 crops with flips (swapping left/right joints), colour
  jitter and rotation.
- **Dense label.** 17 channels, one per joint: a small Gaussian (σ = 3 px) at the joint position.
  Joints the annotator could not see get `M = 0`. The loss is a spatial-softmax cross-entropy, so
  each output channel is a probability map over pixels (peak-normalised for display).
- **Visualisation.** Raw: the 17 heatmaps collapsed with a per-pixel max. Task: each channel's
  argmax is taken as the joint (`dense_to_sparse`) and drawn as a skeleton with the upstream
  `vis_animal_keypoints`, for the annotated joints.

In [ ]:
model, cfg, _ = run_task("animal_keypoint_detection", n_query=4, eval_shot=DEMO_EVAL_SHOT["animal_keypoint_detection"])
del model
su.free_gpu()

### 6.2 Video Object Segmentation (DAVIS-2017)

- **Preprocessing.** Frames are pre-resized to 448×448. The support set is a *single annotated
  frame*, the first one (resized to 384×384); every other frame is a query.
- **Dense label.** One binary channel per tracked instance (the one-hot instance ids, background
  dropped). The loss is a softmax cross-entropy over the instances plus an implicit background
  logit of 0.
- **Visualisation.** Raw: per-instance probabilities `softmax([0, logits])`, one colour per
  instance. Task: the arg-max mask (the learner's `postprocess_final`) overlaid with the upstream
  `postprocess_semseg`.

In [ ]:
model, cfg, _ = run_task("video_object_tracking", n_query=4, eval_shot=DEMO_EVAL_SHOT["video_object_tracking"])
del model
su.free_gpu()

### 6.3 Medical Lesion Segmentation (ISIC-2018)

A domain the meta-training mixture never contained: dermoscopy looks nothing like Taskonomy or COCO.

- **Preprocessing.** Full-resolution dermoscopy images are resized to 448×448; support images to
  384×384. Fine-tuning takes random crops.
- **Dense label.** One binary lesion-mask channel, trained with binary cross-entropy.
- **Visualisation.** Raw: the lesion probability (sigmoid of the output). Task: the mask
  thresholded at 0.5 (`postprocess_final`), overlaid with `postprocess_semseg`.

In [ ]:
model, cfg, _ = run_task("medical_lesion_segmentation", n_query=4, eval_shot=DEMO_EVAL_SHOT["medical_lesion_segmentation"])
del model
su.free_gpu()

### 6.4 Exemplar-Guided Object Counting (FSC-147)

- **Preprocessing.** A two-image task (`M = 2`, so `X` has 6 channels). The photo is resized to
  592×592 (support: 512×512). The second image is the **exemplar channel**: the three exemplar
  boxes are cut out of the photo and pasted repeatedly onto a blank canvas, which tells the model
  *which* object to count. At test time they are tiled at their original scale; during fine-tuning
  at random positions and scales (0.75–1.25×) as augmentation. The first row shows the raw boxes,
  the second row the channel built from them.
- **Dense label.** A density map with one Gaussian per object, so it sums to the count. It is
  divided by its peak to fit `[0, 1]` (the peak is kept to recover the count) and trained with MSE
  on the sigmoid output.
- **Visualisation.** Raw: the density maps. Task: the objects as local maxima of the density
  (upstream `preprocess_kpmap`, threshold 0.2). The label count is the density sum; the predicted
  count is the number of maxima, which is what the evaluation code reports.

In [ ]:
model, cfg, _ = run_task("object_counting", n_query=4, eval_shot=DEMO_EVAL_SHOT["object_counting"])
del model
su.free_gpu()

### 6.5 Cell Instance Segmentation (Cellpose)

- **Preprocessing.** Also a two-image task: the cytoplasm stain and the nuclei stain, each a
  grayscale image repeated to 3 channels. Images keep their native resolution (up to 576×720) and
  are cut into overlapping 224×224 crops (`autocrop`, 50% overlap). The crop predictions are
  blended back together, averaged over horizontal/vertical flips (test-time augmentation).
- **Dense label.** Three channels: the Cellpose flow field (two components pointing toward each
  cell's centre, mapped from `[-1, 1]` to `[0, 1]`) and a binary cell mask. Following the flows
  separates touching cells that a plain mask would merge.
- **Masking out the background (the foreground channel).** The third channel is a *semantic*
  segmentation, cell vs. background, and it decides where the flows mean anything.
  - *How it is obtained:* the dataset gives an **instance mask** per image (`masks.png`, pixel =
    cell id, 0 = background). The foreground label is simply `mask > 0`. The flows are
    precomputed from the same instance mask with Cellpose's heat diffusion (`masks_to_flows` in
    `downstream/cellpose/utils.py`): heat spreads from each cell's centre pixel and its normalised
    gradient becomes the flow, so flows are exactly zero outside cells. The shipped `flows.npy`
    are identical to recomputing them this way.
  - *How it is used:* in training, the foreground channel is supervised like the flows (BCE). At
    inference, only pixels whose predicted cell probability exceeds 0.5 are allowed to follow the
    flows (`compute_masks`). Each such pixel moves along the flow for 200 steps, and pixels that end
    at the same sink become one cell, so background can never turn into a cell. The same
    probability fades the flow colours in the raw row of the figure.
- **Visualisation.** Raw: the flow field as a colour wheel (hue = direction), faded to white where
  the cell probability is low. Task: cell instances recovered by following the flows (Cellpose's
  `compute_masks`), drawn as red boundaries on the red-green stain image (nuclei red, cytoplasm
  green), with the number of cells.

In [ ]:
model, cfg, _ = run_task("cell_instance_segmentation", n_query=4, eval_shot=DEMO_EVAL_SHOT["cell_instance_segmentation"])
del model
su.free_gpu()

---
## Step 7: Your Project — Optional Tracks (+2 pt)

This is where your project starts. On top of it there are two **optional tracks**. You may do
one, both, or neither: completing **either** earns the full **+2 pt bonus**, doing both still
earns 2 pt, and doing neither earns no bonus. If you do one, say which in your write-up and on
your first slide; `README.md` lists what counts as completing it.

> **If you do neither track**, your project must at least **apply the generalist model to
> real-world data**: run a released checkpoint on images you collected or found yourself, outside
> the benchmark datasets, and analyse the results. `su.predict_image`
> (Section 7.6) is the tool for this. Re-running the demo data does not count.

| | **Track A — Fine-tuning** | **Track B — System building** |
| --- | --- | --- |
| you change | the **data and/or task**: new data, a new dense prediction problem, or both, with your own labels | nothing inside the model; you **build something with it** |
| you train | yes — fine-tune Chameleon on your task | optional — use the released checkpoints or one you fine-tuned |
| the result | a working predictor, and an analysis of how well it works | an end-to-end application in which a dense prediction is the core component |
| starts at | 7.1 – 7.5 below | 7.6 below |

## Track A — Fine-tune on your own task

### 7.1 Lay out your data

```
YOUR_DATA_ROOT/
├── train/
│   ├── images/   000.jpg 001.jpg ...
│   └── labels/   000.png 001.png ...    # same filename stem as the image
└── test/
    ├── images/
    └── labels/
```

The first `shot` files in `train/` become the support set — the examples that *define* the task.
Everything in `test/` is a query.

### 7.2 Decide what `Y` means

This is the real design decision. Some recipes:

| your problem | `C` | how to encode it | `loss_type` |
| --- | --- | --- | --- |
| "is this pixel a defect?" | 1 | binary mask in `{0, 1}` | `bce` |
| "which of k materials?" | k | one-hot per class, background dropped | `bce` |
| "where are these k landmarks?" | k | a Gaussian blob per landmark; `M=0` when not visible | `ssl` |
| "how many objects?" | 1 | density map that sums to the count, rescaled to `[0,1]` | `l2` |
| "how far away is this pixel?" | 1 | normalised to `[0,1]` with a fixed, documented range | `l2` |

Two rules that matter more than they look:

- **`Y` must be in `[0, 1]`.** The label encoder treats each channel as an image. If your target is
  unbounded, pick a fixed range, divide, and write that range down — you will need it to invert
  the prediction.
- **Use `M`.** Anywhere the label is unknown rather than zero, set `M = 0`. Training a model to
  predict "nothing" where you simply have no annotation is a silent way to lose accuracy.

### 7.3 Write the loader

Subclass `CustomDenseDataset` and override `load_label`. Usually that is the only method you touch.
If your task has two input modalities (e.g. RGB + thermal), also override `load_image` to return
their channel-wise concatenation, `(6, H, W)`, repeating any single-channel modality to 3.

In [ ]:
from downstream.custom import CustomDenseDataset, CustomLearner, register_custom

class MyDataset(CustomDenseDataset):
    '''TODO: load YOUR labels here.

    Must return (Y, M), both float tensors of shape (C, H, W) with values in [0, 1].
    The example below reads a binary mask PNG -- replace it with whatever your labels look like.
    '''

    def load_label(self, stem):
        from PIL import Image
        label = self.toten(Image.open(self._find(self.label_dir, stem)).convert("L"))
        if label.max() > 1:
            label = label / 255.0
        Y = (label > 0.5).float()          # (1, H, W) binary mask
        M = torch.ones_like(Y)             # every pixel is annotated
        return Y, M


# n_tasks MUST equal the number of channels in your Y.
register_custom(dataset_cls=MyDataset, learner_cls=CustomLearner, n_tasks=1)
print("registered MyDataset for --dataset custom")

### 7.4 Point Chameleon at it and fine-tune

Same loop as Step 4 — only the config changes.

In [ ]:
MY_DATA_ROOT = os.path.join(DRIVE_ROOT, "my_task")   # TODO: your data, laid out as in 7.1

su.write_data_paths(DEMO_DATA_ROOT, extra={"custom": MY_DATA_ROOT}, chameleon_root=CHAMELEON_DIR)

my_cfg = su.build_config(
    stage=1, dataset="custom",
    n_tasks=1,                 # must match register_custom
    loss_type="bce",           # bce | l2 | ssl  -- see the table in 7.2
    shot=20,                   # how many labelled images define the task
    global_batch_size=2,
    n_steps=200,               # TODO: raise once it runs
    lr=0.001,
)

my_model, my_cfg = su.load_chameleon(my_cfg, META_CKPT)
su.trainable_parameter_report(my_model)

In [ ]:
optimizer, lr_scheduler = get_optimizer(my_cfg, my_model.model)
train_loader = my_model.learner.get_train_loader()

my_model.train()
losses, data_iter = [], iter(train_loader)

for step in trange(my_cfg.n_steps, desc="fine-tuning (custom task)"):
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch = next(data_iter)
    batch = to_device(batch, "cuda")

    Y_pred, Y_query, M_query, _ = my_model(batch)
    loss, _ = my_model.learner.compute_loss(Y_pred, Y_query, M_query)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    lr_scheduler.step(step)
    losses.append(loss.item())

my_model.eval()
su.plot_loss(losses, title="fine-tuning on my task")

MY_TASK_CKPT = os.path.join(WORK_DIR, "my_task.ckpt")
su.save_finetuned(my_model, MY_TASK_CKPT)

del my_model, optimizer, lr_scheduler, train_loader, data_iter, batch, Y_pred, Y_query, M_query, loss
su.free_gpu()

### 7.5 Run inference on your queries

In [ ]:
overrides = su.derive_stage2_overrides(MY_TASK_CKPT)
cfg = su.build_config(stage=2, dataset="custom", **overrides)
my_model, cfg = su.load_chameleon(cfg, META_CKPT, MY_TASK_CKPT, verbose=False)
my_model.eval()

samples = su.predict_queries(my_model, n_query=4)
# generic figure: input(s), raw label / prediction, then the learner's postprocess_vis
su.show_predictions(my_model, samples, title="my task")

del my_model
su.free_gpu()

## Track B — Build a system around the model

### 7.6 Use a fine-tuned model as a component

*(Also the starting point if you do neither track: apply a released checkpoint to your own
real-world images.)*

A system needs the model as a plain function, *image in → dense map out*, applied to images that
are not in any dataset. `su.predict_image` does this for the single-image tasks (keypoints, video
masks, lesion masks, and custom RGB tasks). The support set is the one the model was loaded with;
the query is any image you pass. It returns the raw dense output and the decoded prediction, both
at the image's own resolution.

The example below wraps the lesion-segmentation model as a tiny "app function": photo in, overlay
and lesion area out.

In [ ]:
import numpy as np
from PIL import Image

task = "medical_lesion_segmentation"
ft_ckpt = su.find_finetuned(CHECKPOINT_ROOT, task)
overrides = su.derive_stage2_overrides(ft_ckpt, verbose=False)
overrides["shot"] = overrides["eval_shot"] = DEMO_EVAL_SHOT[task]   # T4-sized support set
app_cfg = su.build_config(stage=2, dataset=su.TASKS[task]["dataset"], **overrides)
app_model, app_cfg = su.load_chameleon(app_cfg, META_CKPT, ft_ckpt, verbose=False)
app_model.eval()


def lesion_app(photo):
    '''The "product": a photo in, an overlay and the lesion's share of the image out.'''
    raw, mask = su.predict_image(app_model, photo)
    area = mask.float().mean().item()
    img = torch.from_numpy(np.asarray(photo.convert("RGB"))).permute(2, 0, 1).float() / 255
    overlay = img * (1 - 0.45 * mask) + torch.tensor([1., 0., 0.])[:, None, None] * 0.45 * mask
    return overlay, area


# any RGB image works -- here a query image from the demo folder (index 15 is not in the support set)
photo = Image.open(os.path.join(DEMO_DATA_ROOT, "isic2018", "images", "15.jpg"))
overlay, area = lesion_app(photo)
su.show_images([overlay], titles=[f"lesion covers {100 * area:.1f}% of the image"],
               title="lesion_app(photo)", cols=1, scale=4)

### 7.7 Ideas and practical notes for Track B

- **Conditional image generation.** Dense predictions are exactly what controllable generators
  consume: keypoints, masks and edge maps are ControlNet conditions. `diffusers` provides
  Stable Diffusion + ControlNet pipelines.
- **Web demo.** Wrap a function like `lesion_app` in a Gradio interface and launch it from Colab
  with a public link:
  ```python
  %pip install -qq gradio
  import gradio as gr
  gr.Interface(fn=lambda im: lesion_app(im)[0].permute(1, 2, 0).numpy(),
               inputs=gr.Image(type="pil"), outputs="image").launch(share=True)
  ```
- **Mobile.** Chameleon (~660M parameters) is far too large to run on a phone. Either call a
  server (the Colab Gradio app exposes an API), or **distil**: label a few thousand images with
  Chameleon and train a small on-device model (e.g. a MobileNet U-Net) on those labels. The
  distillation, and how much accuracy it costs, is itself good project material.
- **Memory on a T4.** Chameleon inference alone peaks at 3–11 GB (see the `[GPU]` lines above),
  and a Stable Diffusion + ControlNet pipeline in fp16 needs about 5–7 GB. Run the stages one
  after the other: predict, move the results to CPU, `del` the model and call `su.free_gpu()`,
  then load the next model.
- **What makes a system strong** is that it works end to end on real inputs, that the dense
  prediction is essential rather than decorative, and that you measure it (e.g. latency, memory,
  whether it is actually useful, or a failure case study). See the Track B criteria under
  Evaluation in `README.md`.

In [ ]:
del app_model
su.free_gpu()

---
## Step 8: Tips and Common Problems

**Out of memory.** In the order you should try them:

1. lower `eval_shot` (inference) or `global_batch_size` (fine-tuning);
2. lower `channel_sampling` if your task has many label channels;
3. set `chunk_size=1` and `channel_chunk_size=1` — these make support encoding stream instead of
   batching, trading speed for memory;
4. restart the runtime between tasks; Colab does not always give memory back.

**The prediction is blank or uniform.** Usually one of:

- `Y` is not in `[0, 1]`, so the label encoder saw something it cannot represent;
- `M` is zero everywhere, so the loss had nothing to optimise;
- `n_tasks` does not match `Y.shape[0]`;
- `loss_type` does not match your label type (a density map trained with `bce` will not work).

**The loss drops but the predictions are bad.** Your support set is probably unrepresentative.
Twenty images is enough only if they *cover* the variation — different poses, lighting,
backgrounds. Look at your support set again; it is the whole task specification.

**It works on the support set but not on queries.** That is overfitting, and with 200 steps it is
unlikely — but if you have raised `n_steps` a lot, lower it or add augmentation
(`randomflip`, `randomjitter` in the config).

### Things worth trying for your project

- **A data-efficiency curve.** Rerun with `shot` = 1, 5, 10, 20 and plot quality against it. This
  is the most interesting experiment you can run with this model, and it is cheap.
- **Support-set selection.** Does *which* 20 images you pick matter more than how many? Try
  different `support_idx` values.
- **Compare encodings.** Some tasks can be posed more than one way — e.g. counting as a density
  map versus as blob detection. Which works better, and why?
- **A genuinely new domain.** The meta-training mixture is natural images. How far can you push it
  — thermal, microscopy, satellite, sonar?

---
## What to Submit

See [`README.md`](README.md) for the full requirements. In short, one
`{STUDENT_ID}_{NAME}_project.zip` containing: one `.ipynb` that runs end to end with all outputs
saved, a 2–3 page write-up, your slides, your data as a `.zip` (if you used your own), and, if
you did Track B, any app code that cannot live in the notebook as a `.zip`.

**Grading (17 pt):** write-up 5 pt, presentation 10 pt (peer review), optional track bonus 2 pt.

The write-up should spend most of its space on **the problem and your formulation** — what task
you chose, how you encoded it as a dense output, and what you learned — and, if you did a bonus
track, on your fine-tuning results (Track A) or on what your system does and how well it works
(Track B). Not on restating how Chameleon works.

Good luck! 🦎